# CBAM scope and SBTi coverage for case-origin countries

**What this notebook does.** It characterises two external carbon-regulation datasets in relation to the ShockT cases:

1. **CBAM (EU Carbon Border Adjustment Mechanism):** Do any case goods fall under CBAM? How do case-origin countries compare on embedded emissions for CBAM-covered products?
2. **SBTi (Science Based Targets initiative):** How many companies in case-origin countries have committed to science-based emissions targets?

**Why it matters.** Even though the current ShockT cases (apparel, footwear, electronics) are not CBAM-regulated, case-origin countries' carbon intensity on CBAM products indicates how exposed they are to carbon-pricing regimes. SBTi adoption measures corporate climate-commitment density in the supply chain.

**To run:** set `ANALYST_DATABASE_URL` in `analysis/.env`, then run all cells.

## 1. Setup

In [1]:
import polars as pl

from shockt_analysis import core
from shockt_analysis.settings import get_settings

pl.Config.set_tbl_rows(50)
pl.Config.set_tbl_width_chars(140)
pl.Config.set_float_precision(3)

NAMES = pl.read_csv("../../data-pipelines/dbt/seeds/seed_country_names.csv")

In [2]:
with core.connect(get_settings()) as conn:
    cbam = core.cbam_default_values(conn)
    sbti = core.sbti_companies(conn)
    origins = core.origins(conn)
    cases = core.case_codes(conn)
    loads = core.current_loads(
        conn,
        ["cbam_default_values", "sbti_companies"],
    )

origin_isos = set(origins["iso3"].to_list())

print(f"CBAM: {cbam.height} rows")
print(f"SBTi: {sbti.height} companies")
print(f"Case-origin countries: {len(origin_isos)}")
print()
loads

CBAM: 12540 rows
SBTi: 15909 companies
Case-origin countries: 42



dataset,partitions,latest_load,code_versions
str,i64,str,str
"""cbam_default_values""",1,"""2026-10-29 20:48:37.027522+00""","""565063f"""
"""sbti_companies""",1,"""2026-10-29 20:48:50.567017+00""","""565063f"""


## 2. CBAM: case goods are not covered

CBAM currently covers five sectors: iron and steel, cement, aluminium, fertilisers and hydrogen. The ShockT case HS codes fall in chapters 61-62 (apparel), 64 (footwear) and 84-85 (electronics) — none of which overlap.

In [3]:
cbam_hs4 = set(cbam["cn_code"].str.replace_all(" ", "").str.slice(0, 4).unique().to_list())
case_hs4 = set(p[:4] for p in cases["hs_prefix"].to_list())

cbam_chapters = sorted({c[:2] for c in cbam_hs4})
case_chapters = sorted({c[:2] for c in case_hs4})

print(f"CBAM HS chapters: {cbam_chapters}")
print(f"Case HS chapters: {case_chapters}")
print(f"Overlap: {sorted(case_hs4 & cbam_hs4) or 'none'}")

CBAM HS chapters: ['25', '26', '28', '31', '72', '73', '76']
Case HS chapters: ['61', '62', '64', '84', '85']
Overlap: none


## 3. CBAM: sector summary

Default embedded emissions (t CO₂ / t product) by sector, across all countries.

In [4]:
sector_summary = (
    cbam.group_by("sector")
    .agg(
        pl.col("cn_code").n_unique().alias("n_cn_codes"),
        pl.col("country").n_unique().alias("n_countries"),
        pl.col("total_emissions").drop_nulls().mean().alias("mean_t_co2_per_t"),
        pl.col("total_emissions").drop_nulls().median().alias("median_t_co2_per_t"),
        pl.col("total_emissions").drop_nulls().max().alias("max_t_co2_per_t"),
    )
    .sort("n_cn_codes", descending=True)
)
print("CBAM sector summary (all countries):")
sector_summary

CBAM sector summary (all countries):


sector,n_cn_codes,n_countries,mean_t_co2_per_t,median_t_co2_per_t,max_t_co2_per_t
str,u32,u32,f64,f64,f64
"""Iron and steel""",221,53,3.180,3.000,13.240
"""Fertilisers""",29,91,1.367,1.330,4.810
"""Aluminium""",24,68,1.948,2.031,5.559
"""Cement""",8,101,1.161,1.200,2.290
"""Hydrogen""",1,96,11.656,10.820,26.640


## 4. CBAM: case-origin country carbon intensity

Mean default embedded emissions for case-origin countries that appear in the CBAM dataset, ranked by intensity. Higher values mean more carbon-intensive production, hence greater exposure to future carbon-pricing expansion.

In [5]:
cbam_iso = cbam.join(
    NAMES.select("variant_name", "iso3"),
    left_on="country",
    right_on="variant_name",
    how="left",
)

cbam_origins = cbam_iso.filter(pl.col("iso3").is_in(list(origin_isos)))

by_country = (
    cbam_origins.group_by("iso3")
    .agg(
        pl.col("sector").n_unique().alias("n_sectors"),
        pl.col("cn_code").n_unique().alias("n_cn_codes"),
        pl.col("total_emissions").drop_nulls().len().alias("n_with_values"),
        pl.col("total_emissions").mean().alias("mean_t_co2_per_t"),
    )
    .sort("mean_t_co2_per_t", descending=True)
)

print(f"Case-origin countries with CBAM data: {by_country.height} of {len(origin_isos)}")
by_country

Case-origin countries with CBAM data: 36 of 42


iso3,n_sectors,n_cn_codes,n_with_values,mean_t_co2_per_t
str,u32,u32,u32,f64
"""LAO""",1,1,1,10.820
"""IDN""",4,275,247,7.244
"""BRN""",2,9,2,5.965
"""IND""",5,283,256,4.762
"""CHN""",5,283,259,4.272
"""MEX""",4,275,248,2.723
"""KOR""",5,283,255,2.684
"""BHR""",2,25,25,2.668
"""VNM""",4,275,217,2.561


In [6]:
# Top-5 most carbon-intensive case-origin countries, by sector
top5_isos = by_country.head(5)["iso3"].to_list()

top5_by_sector = (
    cbam_origins.filter(pl.col("iso3").is_in(top5_isos))
    .group_by("iso3", "sector")
    .agg(pl.col("total_emissions").mean().alias("mean_t_co2_per_t"))
    .sort("iso3", "sector")
)

pivoted = top5_by_sector.pivot(on="sector", index="iso3", values="mean_t_co2_per_t")
print("Top-5 most carbon-intensive case-origin countries, by sector:")
pivoted

Top-5 most carbon-intensive case-origin countries, by sector:


iso3,Cement,Hydrogen,Aluminium,Fertilisers,Iron and steel
str,f64,f64,f64,f64,f64
"""BRN""",1.110,10.820,null,null,null
"""CHN""",1.353,26.640,4.928,2.190,4.481
"""IDN""",null,10.820,2.180,1.367,8.662
"""IND""",1.618,14.030,3.535,1.581,5.374
"""LAO""",null,10.820,null,null,null


---

## 5. SBTi: company coverage by case-origin country

How many companies in each case-origin country have set science-based emissions targets?

In [7]:
sbti_iso = sbti.join(
    NAMES.select("variant_name", "iso3"),
    left_on="location",
    right_on="variant_name",
    how="left",
)

matched = sbti_iso.filter(pl.col("iso3").is_not_null()).height
total = sbti_iso.height
print(f"SBTi companies matched to ISO3: {matched} of {total} ({matched / total * 100:.0f}%)")

unmatched = sbti_iso.filter(pl.col("iso3").is_null())["location"].unique().sort().to_list()
if unmatched:
    print(f"Unmatched locations ({len(unmatched)}): {unmatched}")

SBTi companies matched to ISO3: 15909 of 15909 (100%)


In [8]:
sbti_origins = sbti_iso.filter(pl.col("iso3").is_in(list(origin_isos)))

sbti_by_country = (
    sbti_origins.group_by("iso3")
    .agg(
        pl.len().alias("n_companies"),
        (pl.col("near_term_status") == "Validated Targets").sum().alias("validated"),
        (pl.col("near_term_status") == "Commitment to Set Targets").sum().alias("committed"),
        (pl.col("near_term_status") == "Commitment Removed").sum().alias("removed"),
    )
    .sort("n_companies", descending=True)
)

total_origin = sbti_by_country["n_companies"].sum()
total_global = sbti_iso.filter(pl.col("iso3").is_not_null()).height

print(
    f"SBTi companies in case-origin countries: {total_origin} "
    f"of {total_global} globally ({total_origin / total_global * 100:.1f}%)"
)
print(f"Countries with SBTi companies: {sbti_by_country.height} of {len(origin_isos)}")
print()
sbti_by_country

SBTi companies in case-origin countries: 5556 of 15909 globally (34.9%)
Countries with SBTi companies: 30 of 42



iso3,n_companies,validated,committed,removed
str,u32,u32,u32,u32
"""JPN""",2680,2598,60,22
"""CHN""",1094,843,165,86
"""IND""",607,386,123,98
"""TWN""",234,185,32,17
"""KOR""",129,90,22,17
"""HKG""",120,97,14,9
"""SGP""",103,69,16,18
"""BGD""",97,65,13,19
"""PAK""",80,36,13,31


In [9]:
# SBTi sector breakdown for case-origin countries
sbti_sectors = (
    sbti_origins.group_by("sector")
    .agg(
        pl.len().alias("n_companies"),
        (pl.col("near_term_status") == "Validated Targets").sum().alias("validated"),
    )
    .sort("n_companies", descending=True)
)
print("SBTi companies in case-origin countries, by sector (top 15):")
sbti_sectors.head(15)

SBTi companies in case-origin countries, by sector (top 15):


sector,n_companies,validated
str,u32,u32
"""Electrical Equipment and Machi…",796,727
"""Construction and Engineering""",483,465
"""Textiles, Apparel, Footwear an…",452,288
"""Automobiles and Components""",324,255
"""Trading Companies and Distribu…",318,288
"""Consumer Durables, Household a…",298,270
"""Professional Services""",291,262
"""Technology Hardware and Equipm…",234,168
"""Software and Services""",230,189


## 6. Findings

In [10]:
validated_origin = sbti_by_country["validated"].sum()
validated_pct = validated_origin / total_origin * 100

findings = [
    "No overlap: current ShockT case goods (HS chapters 61-62, "
    "64, 84-85) are not covered by CBAM (chapters 25-26, 28, "
    "31, 72-73, 76). CBAM targets heavy industry (steel, cement, "
    "aluminium, fertilisers, hydrogen), not consumer goods.",
    f"Of {len(origin_isos)} case-origin countries, {by_country.height} "
    "appear in the CBAM dataset. Their mean default embedded "
    "emissions range from "
    f"{by_country['mean_t_co2_per_t'].min():.1f} to "
    f"{by_country['mean_t_co2_per_t'].max():.1f} t CO₂/t product.",
    "The most carbon-intensive case-origin countries on CBAM "
    "products are "
    + ", ".join(by_country.head(3)["iso3"].to_list())
    + ". High intensity signals exposure to future carbon-"
    "pricing expansion beyond CBAM's current scope.",
    f"SBTi: {total_origin} companies in case-origin countries "
    f"({total_origin / total_global * 100:.1f}% of {total_global} globally). "
    f"{validated_origin} ({validated_pct:.0f}%) have validated near-term "
    f"targets. {sbti_by_country.head(1)['iso3'][0]} leads with "
    f"{sbti_by_country.head(1)['n_companies'][0]} companies.",
    f"{sbti_by_country.height} of {len(origin_isos)} case-origin "
    "countries have at least one SBTi company. Textiles, Apparel, "
    "Footwear — directly relevant to Case 1 — is the 3rd-largest "
    "SBTi sector in these countries.",
]
for i, f in enumerate(findings, 1):
    print(f"{i}. {f}")

1. No overlap: current ShockT case goods (HS chapters 61-62, 64, 84-85) are not covered by CBAM (chapters 25-26, 28, 31, 72-73, 76). CBAM targets heavy industry (steel, cement, aluminium, fertilisers, hydrogen), not consumer goods.
2. Of 42 case-origin countries, 36 appear in the CBAM dataset. Their mean default embedded emissions range from 0.9 to 10.8 t CO₂/t product.
3. The most carbon-intensive case-origin countries on CBAM products are LAO, IDN, BRN. High intensity signals exposure to future carbon-pricing expansion beyond CBAM's current scope.
4. SBTi: 5556 companies in case-origin countries (34.9% of 15909 globally). 4605 (83%) have validated near-term targets. JPN leads with 2680 companies.
5. 30 of 42 case-origin countries have at least one SBTi company. Textiles, Apparel, Footwear — directly relevant to Case 1 — is the 3rd-largest SBTi sector in these countries.
